##   **Task 1 – Data Integrity & Quality Validation with Business Logic**
###  Objectives
## 1. Validate data integrity and quality checks across Bronze layer datasets.
## 2. Implement and review business logic validations for correctness and completeness.

## Architecture Overview

This notebook implements data quality validation for the Bronze layer within a Medallion Architecture framework.

### Data Processing Flow

Source Data (CSV files in Data Lake)  
↓  
Bronze Layer (Raw Delta Tables)  
↓  
Data Profiling  
↓  
Data Quality Validation  
↓  
Business Logic Validation  
↓  
Governance Monitoring & Audit Logging  
↓  
Silver Layer (Downstream Cleansing)

The Bronze layer preserves raw source data while identifying data quality exceptions that will be handled in downstream processing layers.

**ENVIRONMENT SETUP (FIRST CELL)**

In [0]:
 -- Set Working Environment
USE CATALOG man_cata;
USE SCHEMA default;

## Data Quality Validation Framework

To ensure reliability of the Bronze datasets, a structured data quality framework has been implemented.

The validation approach follows industry-standard data quality pillars:

• Completeness – Required fields must not contain NULL values  
• Uniqueness – Entity identifiers and composite keys must remain unique  
• Consistency – Referential relationships across datasets must remain valid  
• Validity – Data values must fall within defined business ranges  
• Accuracy – Financial and operational data must align with business logic  
• Timeliness – Lifecycle events must follow correct chronological order  

These validations ensure that the Bronze layer maintains raw data integrity while detecting anomalies for downstream remediation.

In [0]:
-- Verify Raw Files in Data Lake
LIST '/Volumes/man_cata/default/olist_volume/';

## Bronze Layer – Raw Data Ingestion

In [0]:
CREATE OR REPLACE TEMP VIEW orders_raw
USING CSV
OPTIONS (
  path "/Volumes/man_cata/default/olist_volume/olist_orders_dataset.csv",
  header "true",
  inferSchema "true"
);

In [0]:
CREATE OR REPLACE TABLE bronze_orders
USING DELTA
AS
SELECT * FROM orders_raw;

In [0]:
DESCRIBE bronze_orders;

In [0]:
CREATE OR REPLACE TEMP VIEW customers_raw
USING CSV
OPTIONS (
 path "/Volumes/man_cata/default/olist_volume/olist_customers_dataset.csv",
 header "true",
 inferSchema "true"
);

CREATE OR REPLACE TABLE bronze_customers
USING DELTA
AS SELECT * FROM customers_raw;

In [0]:
CREATE OR REPLACE TEMP VIEW order_items_raw
USING CSV
OPTIONS (
 path "/Volumes/man_cata/default/olist_volume/olist_order_items_dataset.csv",
 header "true",
 inferSchema "true"
);

CREATE OR REPLACE TABLE bronze_order_items
USING DELTA
AS SELECT * FROM order_items_raw;

In [0]:
CREATE OR REPLACE TEMP VIEW payments_raw
USING CSV
OPTIONS (
 path "/Volumes/man_cata/default/olist_volume/olist_order_payments_dataset.csv",
 header "true",
 inferSchema "true"
);

CREATE OR REPLACE TABLE bronze_payments
USING DELTA
AS SELECT * FROM payments_raw;

In [0]:
CREATE OR REPLACE TEMP VIEW products_raw
USING CSV
OPTIONS (
 path "/Volumes/man_cata/default/olist_volume/olist_products_dataset.csv",
 header "true",
 inferSchema "true"
);

CREATE OR REPLACE TABLE bronze_products
USING DELTA
AS SELECT * FROM products_raw;

In [0]:
CREATE OR REPLACE TEMP VIEW sellers_raw
USING CSV
OPTIONS (
 path "/Volumes/man_cata/default/olist_volume/olist_sellers_dataset.csv",
 header "true",
 inferSchema "true"
);

CREATE OR REPLACE TABLE bronze_sellers
USING DELTA
AS SELECT * FROM sellers_raw;

In [0]:
CREATE OR REPLACE TEMP VIEW reviews_raw
USING CSV
OPTIONS (
 path "/Volumes/man_cata/default/olist_volume/olist_order_reviews_dataset.csv",
 header "true",
 inferSchema "true"
);

CREATE OR REPLACE TABLE bronze_reviews
USING DELTA
AS SELECT * FROM reviews_raw;

In [0]:
SHOW TABLES;

## Dataset Relationship Overview

The ecommerce dataset contains transactional and dimensional datasets with the following relationships:

- Customers → Orders → Order Items  
- Orders → Payments  
- Order Items → Products  
- Order Items → Sellers  
- Orders → Reviews  

These relationships form the transactional ecosystem of the ecommerce platform and are validated using referential integrity checks.

# Data Profiling
# 

### Data Profiling Objective

Data profiling is performed to understand the structure, size, and characteristics of the datasets before applying validation rules.

Profiling includes:

• Volume analysis (row counts)  
• Schema inspection (column types)  
• Initial structural validation  

This step ensures the ingestion process loaded the expected datasets correctly before applying deeper data quality validations.

In [0]:
SELECT COUNT(*)
FROM man_cata.default.bronze_orders;

## Volume Profiling

In [0]:
-- Transaction Tables
SELECT COUNT(*) AS orders FROM bronze_orders;
SELECT COUNT(*) AS order_items FROM bronze_order_items;
SELECT COUNT(*) AS payments FROM bronze_payments;
SELECT COUNT(*) AS reviews FROM bronze_reviews;

-- Dimension Tables
SELECT COUNT(*) AS customers FROM bronze_customers;
SELECT COUNT(*) AS products FROM bronze_products;
SELECT COUNT(*) AS sellers FROM bronze_sellers;

## Schema Profiling

In [0]:
-- Transaction Tables
DESCRIBE bronze_orders;
DESCRIBE bronze_order_items;
DESCRIBE bronze_payments;
DESCRIBE bronze_reviews;

-- Dimension Tables
DESCRIBE bronze_customers;
DESCRIBE bronze_products;
DESCRIBE bronze_sellers;

## Completeness Profiling

In [0]:
SELECT COUNT(*)
FROM bronze_orders
WHERE order_id IS NULL;

In [0]:
SELECT COUNT(*) FROM bronze_order_items WHERE order_id IS NULL;

In [0]:
SELECT COUNT(*) FROM bronze_payments WHERE order_id IS NULL;

In [0]:
SELECT COUNT(*) FROM bronze_reviews WHERE order_id IS NULL;

In [0]:
SELECT COUNT(*) FROM bronze_customers WHERE customer_id IS NULL;

In [0]:
SELECT COUNT(*) FROM bronze_products WHERE product_id IS NULL;

In [0]:
SELECT COUNT(*) FROM bronze_sellers WHERE seller_id IS NULL;

## Uniqueness Profiling

In [0]:
SELECT order_id, COUNT(*)
FROM bronze_orders
GROUP BY order_id
HAVING COUNT(*) > 1;

No duplicate customer identifiers detected, confirming entity uniqueness within customer master dataset.

In [0]:
SELECT product_id, COUNT(*)
FROM bronze_products
GROUP BY product_id
HAVING COUNT(*) > 1;

No duplicate product identifiers detected, confirming uniqueness within the product master dataset.

In [0]:
SELECT seller_id, COUNT(*)
FROM bronze_sellers
GROUP BY seller_id
HAVING COUNT(*) > 1;

No duplicate seller identifiers detected, confirming uniqueness within the seller master dataset.

In [0]:
SELECT review_id, COUNT(*)
FROM bronze_reviews
GROUP BY review_id
HAVING COUNT(*) > 1;

In [0]:
DESCRIBE bronze_reviews;

In [0]:
SELECT *
FROM bronze_reviews
LIMIT 10;

Duplicate values observed in review_id indicate that the column does not function as a unique business identifier. Based on dataset analysis, review records are validated using order_id relationship instead.

In [0]:
SELECT order_id, order_item_id, COUNT(*)
FROM bronze_order_items
GROUP BY order_id, order_item_id
HAVING COUNT(*) > 1;

No duplicate order item records detected based on the composite key (order_id, order_item_id), confirming transactional item-level uniqueness.

In [0]:
SELECT order_id, payment_sequential, COUNT(*)
FROM bronze_payments
GROUP BY order_id, payment_sequential
HAVING COUNT(*) > 1;

No duplicate payment records detected based on the composite key (order_id, payment_sequential), confirming transactional payment integrity

REFERENTIAL INTEGRITY VALIDATION

## REFERENTIAL INTEGRITY VALIDATION

In [0]:
SELECT o.order_id
FROM bronze_orders o
LEFT JOIN bronze_customers c
ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

No orphan order records were identified. All orders are successfully associated with valid customers, confirming referential integrity between orders and customers datasets.

In [0]:
SELECT p.order_id
FROM bronze_payments p
LEFT JOIN bronze_orders o
ON p.order_id = o.order_id
WHERE o.order_id IS NULL;

All payment transactions reference valid orders. No orphan payment records were detected, confirming transactional consistency between payments and orders.

In [0]:
SELECT oi.order_id
FROM bronze_order_items oi
LEFT JOIN bronze_orders o
ON oi.order_id = o.order_id
WHERE o.order_id IS NULL;

Every order item record is linked to an existing order, ensuring item-level transactional integrity within the dataset.

In [0]:
SELECT oi.product_id
FROM bronze_order_items oi
LEFT JOIN bronze_products p
ON oi.product_id = p.product_id
WHERE p.product_id IS NULL;

All order items reference valid products from the product master dataset, confirming consistency between transactional and product reference data.

In [0]:
SELECT oi.seller_id
FROM bronze_order_items oi
LEFT JOIN bronze_sellers s
ON oi.seller_id = s.seller_id
WHERE s.seller_id IS NULL;

No invalid seller references were identified. Each order item is associated with a valid seller entity.

In [0]:
SELECT r.order_id
FROM bronze_reviews r
LEFT JOIN bronze_orders o
ON r.order_id = o.order_id
WHERE o.order_id IS NULL;

In [0]:
SELECT *
FROM bronze_reviews
LIMIT 20;

Referential integrity validation identified malformed review records containing null or invalid order identifiers. The issue appears to originate from CSV parsing inconsistencies during source ingestion. These records have been flagged as data quality exceptions for downstream cleansing in subsequent processing layers.

CHECK CONSTRAINT VALIDATION (DATA QUALITY RULES)

Payment Value Validation

In [0]:
SELECT *
FROM bronze_payments
WHERE payment_value < 0;

Payment validation identified transactions with zero payment values associated with voucher-based or fully discounted orders. Since zero-value payments represent valid business scenarios, validation logic was refined to ensure only negative payment amounts are treated as data quality violations.

Review Score Validation

In [0]:
SELECT *
FROM bronze_reviews
WHERE TRY_CAST(review_score AS INT) NOT BETWEEN 1 AND 5;

Review score validation identified malformed records where review_score values fall outside the expected business range (1–5). Investigation indicates CSV parsing inconsistencies leading to column misalignment during ingestion. These records have been classified as data quality exceptions and will be addressed during downstream data cleansing in the Silver processing layer.

Product Price Validation

In [0]:
SELECT *
FROM bronze_order_items
WHERE price < 0;

No negative product prices identified, confirming correctness of transactional pricing data.

Freight Value Validation

In [0]:
SELECT *
FROM bronze_order_items
WHERE freight_value < 0;

Freight values are valid and non-negative across all order item records.

LIFECYCLE VALIDATION

Lifecycle Validation – Order Business Process Validation

Lifecycle validation ensures chronological correctness of transactional events within the ecommerce order lifecycle. 

The objective is to confirm that business events such as purchase, shipment, and delivery occur in logically consistent temporal order.

In [0]:
-- Delivery cannot happen before purchase

SELECT COUNT(*) AS invalid_delivery_records
FROM bronze_orders
WHERE order_delivered_customer_date < order_purchase_timestamp;

No lifecycle violations detected where delivery occurred prior to purchase. 
This confirms chronological consistency within customer order fulfillment processes.

In [0]:
-- Estimated delivery must occur after purchase

SELECT COUNT(*) AS invalid_estimated_delivery
FROM bronze_orders
WHERE order_estimated_delivery_date < order_purchase_timestamp;

Estimated delivery timelines align correctly with purchase timestamps, confirming valid system-generated delivery projections.

In [0]:
-- Negative payment validation

SELECT COUNT(*) AS negative_payments
FROM bronze_payments
WHERE payment_value < 0;

In [0]:
-- Invalid installment validation

SELECT COUNT(*) AS invalid_installments
FROM bronze_payments
WHERE payment_installments < 0;

In [0]:
-- Invalid product pricing

SELECT COUNT(*) AS invalid_price_records
FROM bronze_order_items
WHERE price <= 0;

Financial lifecycle validation confirms that payment values, installment counts, and product pricing adhere to expected business rules. No invalid financial transactions were identified.

Final Data Quality Validation Summary

Comprehensive data integrity and quality validations were performed across all Bronze layer datasets.

The validation framework included:

• Data Profiling and Volume Analysis  
• Structural Integrity Validation  
• Referential Integrity Checks  
• Business Rule and Check Constraint Validation  
• Financial Consistency Validation  
• Order Lifecycle Validation  

Key Findings:

• No structural or referential inconsistencies detected across transactional datasets.  
• Financial and lifecycle validations confirm adherence to expected ecommerce business processes.  
• Minor data quality exceptions were identified within review datasets due to source CSV parsing inconsistencies and have been flagged for downstream cleansing.

Conclusion:

The Bronze layer successfully preserves raw source data while identifying quality exceptions for controlled remediation in subsequent Silver transformations, aligning with Medallion Architecture best practices.

# BUSINESS LOGIC VALIDATION

FINANCIAL RECONCILIATION

In [0]:
CREATE OR REPLACE VIEW vw_order_value AS
SELECT
    order_id,
    SUM(price + freight_value) AS calculated_order_value
FROM bronze_order_items
GROUP BY order_id;

The total order value has been calculated by aggregating item-level price and freight charges from the transactional order items dataset.
This step establishes an authoritative financial baseline representing the expected customer payable amount for each order.
The derived order value will be utilized in downstream financial reconciliation to validate payment consistency and detect potential revenue mismatches or transactional anomalies.

# Business Logic Validation – Financial Reconciliation

-- 
Financial reconciliation validation ensures consistency between
expected order revenue derived from transactional item data and
actual customer payments recorded in the payment system.

This step validates financial correctness and helps identify
missing, partial, or excess payment scenarios

In [0]:
CREATE OR REPLACE VIEW vw_payment_value AS
SELECT
    order_id,
    SUM(payment_value) AS total_payment_value
FROM bronze_payments
GROUP BY order_id;

Customer payments were aggregated at the order level to account
for installment-based or multi-method transactions. The aggregated
payment value represents total monetary inflow per order.

In [0]:
SELECT 
    o.order_id,
    ROUND(o.calculated_order_value,2) AS calculated_order_value,
    ROUND(p.total_payment_value,2) AS total_payment_value,
    ROUND(
        o.calculated_order_value - p.total_payment_value,
        2
    ) AS difference
FROM vw_order_value o
LEFT JOIN vw_payment_value p
ON o.order_id = p.order_id
WHERE ABS(
        ROUND(o.calculated_order_value,2) -
        ROUND(p.total_payment_value,2)
      ) > 0.01;

Order-level financial reconciliation was performed by comparing calculated order values with aggregated payment transactions to validate data integrity and consistency.

Minor discrepancies were identified and flagged through implemented business logic checks to ensure correctness, completeness, and financial reporting accuracy.

Order Completion Validation

In [0]:
SELECT 
    o.order_id,
    o.order_status
FROM bronze_orders o
LEFT JOIN bronze_payments p
ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
AND p.order_id IS NULL;

Operational validation identified delivered orders without corresponding payment records. According to ecommerce business rules, an order should only reach the delivered state after successful payment confirmation.

The detected record represents a potential operational or data consistency issue that may lead to revenue loss or incorrect order lifecycle tracking.

Delivery Performance Logic

In [0]:
SELECT
    order_id,
    order_estimated_delivery_date,
    order_delivered_customer_date,
    CASE
        WHEN order_delivered_customer_date 
             <= order_estimated_delivery_date
        THEN 'On Time'
        ELSE 'Delayed'
    END AS delivery_status
FROM bronze_orders
WHERE order_delivered_customer_date IS NOT NULL;

### Delivery Performance Business Logic Validation Result

Delivery performance validation was conducted by comparing actual customer delivery dates with estimated delivery timelines. Orders delivered within the promised timeframe were classified as On Time, while orders delivered beyond the estimated date were identified as Delayed.

This validation enables monitoring of logistics efficiency and supports operational performance analysis for order fulfillment processes.

Order Lifecycle Validation

In [0]:
SELECT
    order_id,
    CASE
        WHEN order_delivered_customer_date IS NOT NULL
            THEN 'Completed'
        WHEN order_approved_at IS NOT NULL
            THEN 'Approved'
        ELSE 'Pending'
    END AS business_status
FROM bronze_orders;

Order lifecycle stages were derived using operational timestamps to represent business process progression. Orders with confirmed delivery were classified as Completed, approved orders were categorized as Approved, and remaining orders were identified as Pending.

This transformation converts system-level event data into business-readable operational states, enabling effective monitoring of order fulfillment workflows.

##Order Status Distribution Analysis

In [0]:
SELECT
    order_status,
    COUNT(*) AS total_orders
FROM bronze_orders
GROUP BY order_status
ORDER BY total_orders DESC;

### Order Status Distribution Analysis

Order status distribution was analyzed to evaluate overall operational performance and fulfillment efficiency. The analysis highlights the proportion of orders across different lifecycle stages such as delivered, shipped, canceled, and processing.

A high volume of delivered orders indicates successful order fulfillment, while canceled and unavailable orders may represent operational or inventory-related challenges requiring further investigation.

##Shipment Delay

In [0]:
SELECT
    order_id,
    order_approved_at,
    order_delivered_carrier_date,
    DATEDIFF(
        order_delivered_carrier_date,
        order_approved_at
    ) AS shipment_delay_days
FROM bronze_orders
WHERE order_approved_at IS NOT NULL
AND order_delivered_carrier_date IS NOT NULL
AND DATEDIFF(
        order_delivered_carrier_date,
        order_approved_at
    ) > 2;

### Shipment Processing Delay Check

This validation was performed to check how long it takes for an order to be shipped after approval. Ideally, orders should be handed over to the delivery partner within a short time after approval.

Orders taking more than expected processing time were identified as delayed shipments. This helps highlight possible warehouse or operational delays that may affect overall delivery timelines.

##Order Fulfillment Time

In [0]:
SELECT
    order_id,
    order_purchase_timestamp,
    order_delivered_customer_date,
    DATEDIFF(
        order_delivered_customer_date,
        order_purchase_timestamp
    ) AS fulfillment_days
FROM bronze_orders
WHERE order_purchase_timestamp IS NOT NULL
AND order_delivered_customer_date IS NOT NULL
AND DATEDIFF(
        order_delivered_customer_date,
        order_purchase_timestamp
    ) > 7;

### Order Fulfillment Time Check

This check was performed to understand the total time taken to complete an order from purchase to final delivery. Orders taking longer than the expected fulfillment time were identified to highlight possible delays across order processing, shipment, or delivery stages.

## Business Logic Validation Summary

Business logic validations were implemented to ensure that
financial transactions, operational workflows, and logistics
processes accurately represent real ecommerce business behavior.

The validation framework covered:

• Financial reconciliation between orders and payments  
• Order lifecycle and completion validation  
• Delivery performance monitoring  
• Shipment processing efficiency  
• End-to-end order fulfillment timelines  

These checks confirm operational consistency,
financial correctness, and customer delivery performance
across the transactional ecosystem.

##Data Quality Governance & Validation Monitoring Framework

To ensure traceability and monitoring of data quality and business validation checks, an audit logging framework was implemented.

Instead of manually reviewing validation outputs, validation results are captured and stored in an audit table to maintain execution history and enable governance monitoring.

## Operational Monitoring & SLA Awareness

In production data pipelines, datasets must be delivered within defined Service Level Agreements (SLAs).

Data quality monitoring plays a critical role in detecting issues that may impact downstream data availability.

Validation results captured in the audit table enable:

• early detection of data anomalies  
• operational monitoring of data pipelines  
• investigation of potential SLA breaches  
• historical tracking of data quality trends


Audit Table (Governance Layer)

In [0]:
CREATE OR REPLACE TABLE validation_audit (
    rule_name STRING,
    issue_count INT,
    run_timestamp TIMESTAMP
)
USING DELTA;

The audit table records validation execution results including rule name, number of detected issues, and execution timestamp.

This enables historical tracking of data quality trends and supports operational monitoring in production environments.

##Financial Reconciliation Audit

## Financial Reconciliation – Audit Logging

**Objective**  
To validate financial consistency between calculated order value (items + freight) and recorded customer payments.

**Approach**  
Order-level revenue was derived from transactional order items and compared against aggregated payment transactions.  
Orders with a monetary difference greater than 0.01 were classified as reconciliation mismatches.

The total number of mismatched orders was logged into the validation audit table along with execution timestamp.

**Observation**  
Discrepancies were identified between expected revenue and recorded payment amounts.

**Business Impact**  
Financial mismatches may indicate partial payments, multi-installment timing differences, rounding inconsistencies, or data synchronization gaps.  
Logging these results enables traceability, governance monitoring, and historical trend analysis of financial data quality.

In [0]:
INSERT INTO validation_audit
SELECT
    'Financial_Reconciliation',
    COUNT(*),
    current_timestamp()
FROM (
    SELECT o.order_id
    FROM vw_order_value o
    LEFT JOIN vw_payment_value p
        ON o.order_id = p.order_id
    WHERE p.total_payment_value IS NULL
       OR ABS(o.calculated_order_value - p.total_payment_value) > 0.01
);

##Delivered Without Payment

## Delivered Without Payment – Audit Logging

**Objective**  
To identify orders marked as *delivered* that do not have any corresponding payment records.

**Approach**  
Delivered orders were left-joined with the payments dataset to detect cases where no payment reference exists.  
Orders with status = 'delivered' and missing payment records were classified as potential inconsistencies.

The total number of such cases was logged into the validation audit table along with execution timestamp for governance tracking.

**Observation**  
A small number of delivered orders were identified without associated payment entries.

**Business Impact**  
This scenario may indicate payment synchronization delays, data ingestion timing differences, or operational exceptions.  
Logging these cases ensures traceability and enables ongoing monitoring of financial and lifecycle consistency.

In [0]:
INSERT INTO validation_audit
SELECT
    'Delivered_Without_Payment',
    COUNT(*),
    current_timestamp()
FROM (
    SELECT o.order_id
    FROM bronze_orders o
    LEFT JOIN bronze_payments p
    ON o.order_id = p.order_id
    WHERE o.order_status='delivered'
    AND p.order_id IS NULL
);

##Shipment Delay Audit


Shipment Delay – Audit Logging
Objective
To identify orders where shipment to the carrier was delayed beyond the defined operational threshold after order approval.

Approach
Shipment delay was calculated as the difference between order_delivered_carrier_date and order_approved_at.

Orders where the delay exceeded 2 days were classified as shipment delays.

The total number of delayed orders was logged into the validation_audit table along with the execution timestamp.

Observation
A significant number of orders exceeded the expected shipment processing time.

Business Impact
Shipment delays may indicate warehouse bottlenecks, operational inefficiencies, or logistics coordination gaps.
Audit logging enables governance monitoring and trend analysis of operational performance across runs.

In [0]:
INSERT INTO validation_audit
SELECT
    'Shipment_Delay',
    COUNT(*),
    current_timestamp()
FROM (
    SELECT order_id
    FROM bronze_orders
    WHERE order_approved_at IS NOT NULL
    AND order_delivered_carrier_date IS NOT NULL
    AND DATEDIFF(order_delivered_carrier_date,
                 order_approved_at) > 2
);

##Fulfillment Delay Audit

## Order Fulfillment Delay – Audit Logging

**Objective**  
To identify orders where total fulfillment time exceeded the defined service threshold from purchase to final delivery.

**Approach**  
Fulfillment time was calculated as the difference between `order_delivered_customer_date` and `order_purchase_timestamp`.

Orders with fulfillment duration greater than 7 days were classified as delayed fulfillments.

The total number of such cases was logged into the `validation_audit` table along with the execution timestamp.

**Observation**  
Orders exceeding the expected fulfillment timeline were detected.

**Business Impact**  
Extended fulfillment duration may indicate delays across order processing, shipment handling, or last-mile delivery.  
Audit logging enables performance monitoring and supports operational improvement initiatives.

In [0]:
INSERT INTO validation_audit
SELECT
    'Fulfillment_Delay',
    COUNT(*),
    current_timestamp()
FROM (
    SELECT order_id
    FROM bronze_orders
    WHERE order_purchase_timestamp IS NOT NULL
    AND order_delivered_customer_date IS NOT NULL
    AND DATEDIFF(order_delivered_customer_date,
                 order_purchase_timestamp) > 7
);

##Delivery Performance (Delayed Orders)

## Delivery Performance – Delayed Orders Audit

**Objective**  
To identify orders that were delivered after the promised estimated delivery date.

**Approach**  
Actual delivery dates (`order_delivered_customer_date`) were compared against the estimated delivery timelines (`order_estimated_delivery_date`).

Orders where the actual delivery date exceeded the estimated delivery date were classified as delayed deliveries.

The total number of delayed orders was logged into the `validation_audit` table along with execution timestamp for governance monitoring.

**Observation**  
Orders exceeding the committed delivery timeline were identified.

**Business Impact**  
Delayed deliveries may impact customer satisfaction, service-level adherence, and brand reliability.  
Audit logging enables continuous monitoring of delivery performance and supports logistics optimization initiatives.

In [0]:
INSERT INTO validation_audit
SELECT
    'Delayed_Delivery',
    COUNT(*),
    current_timestamp()
FROM bronze_orders
WHERE order_delivered_customer_date IS NOT NULL
AND order_estimated_delivery_date IS NOT NULL
AND order_delivered_customer_date > order_estimated_delivery_date;

##Lifecycle Violation Audit

## Order Lifecycle Violation – Audit Logging

**Objective**  
To validate chronological consistency within the order lifecycle and ensure that delivery events do not occur before purchase events.

**Approach**  
The actual customer delivery date (`order_delivered_customer_date`) was compared against the order purchase timestamp (`order_purchase_timestamp`).

Orders where the delivery date occurred earlier than the purchase timestamp were classified as lifecycle violations.

The total number of such violations was logged into the `validation_audit` table along with execution timestamp for governance monitoring.

**Observation**  
Chronological inconsistencies were identified within the order lifecycle data.

**Business Impact**  
Lifecycle violations may indicate data ingestion errors, timestamp inconsistencies, or upstream system synchronization issues.  
Audit logging ensures traceability and supports data integrity governance across transactional processes.

In [0]:
INSERT INTO validation_audit
SELECT
    'Lifecycle_Violation',
    COUNT(*),
    current_timestamp()
FROM bronze_orders
WHERE order_delivered_customer_date IS NOT NULL
AND order_purchase_timestamp IS NOT NULL
AND order_delivered_customer_date < order_purchase_timestamp;

In [0]:
SELECT COUNT(*) AS total_validation_rules
FROM validation_audit;

Validation monitoring confirms that 9 data quality and business logic
validation rules were successfully executed and logged in the
validation_audit table.

The audit framework ensures traceability of validation outcomes,
enables governance monitoring, and supports historical analysis
of operational data quality trends across pipeline executions.

In [0]:
SELECT * 
FROM validation_audit
ORDER BY run_timestamp DESC;

### Governance Monitoring Result

The validation audit log provides visibility into all executed data quality
and business validation rules. Each rule execution records the detected
issue count along with execution timestamp, enabling operational monitoring,
traceability, and historical analysis of data quality trends across pipeline runs.

## Data Quality Metrics Overview

The validation framework measures data quality across multiple dimensions:

Completeness → Missing key identifiers  
Uniqueness → Duplicate entity detection  
Consistency → Referential integrity validation  
Validity → Range and format checks  
Accuracy → Financial reconciliation validation  
Timeliness → Lifecycle event sequencing

These metrics provide a comprehensive assessment of data reliability within the Bronze layer.

## Production Readiness Considerations

The implemented validation framework supports production-grade data pipelines by enabling:

• automated data quality monitoring  
• audit trail of validation outcomes  
• detection of operational anomalies  
• governance visibility into pipeline health

This ensures that downstream transformations operate on reliable and traceable datasets.

## Week 1 – Data Integrity & Quality Implementation Summary

Week 1 focused on establishing the Bronze data layer and implementing a structured data quality validation framework.

Key implementation areas included:

• Raw dataset ingestion from the data lake into Delta Bronze tables  
• Data profiling for structural and volume validation  
• Data quality validations across completeness, uniqueness, consistency, and validity pillars  
• Referential integrity validation across transactional and dimension datasets  
• Business logic validations including financial reconciliation and order lifecycle checks  
• Operational performance validations such as shipment delay and fulfillment monitoring  
• Governance monitoring through validation audit logging

These validations ensure that the Bronze layer preserves raw source data while identifying quality exceptions for controlled remediation in downstream Silver transformations following Medallion Architecture best practices.

# Task 2 – SCD, Transformations & Validation in PySpark
## Objective
### 1.Implement Slowly Changing Dimension (SCD) logic in PySpark.
### 2.Perform data transformations and validation using PySpark. 

## Environment Setup


## Configure catalog and schema used in Week-1 Bronze pipeline.

In [0]:
%python

from pyspark.sql.functions import *
from delta.tables import *

spark.sql("USE CATALOG man_cata")
spark.sql("USE SCHEMA default")

print("Environment configured successfully")

## Load Bronze Layer Tables

Bronze tables created in Week-1 act as the source layer for Silver transformations.

In [0]:
%python
orders_df = spark.table("bronze_orders")
customers_df = spark.table("bronze_customers")
products_df = spark.table("bronze_products")
sellers_df = spark.table("bronze_sellers")
order_items_df = spark.table("bronze_order_items")
payments_df = spark.table("bronze_payments")
reviews_df = spark.table("bronze_reviews")

## Bronze Data Profiling

Before performing data integrity and validation checks, we verify the number of records present in each Bronze table.

This step ensures that the data was successfully ingested during the Week-1 pipeline and helps us understand the dataset size before applying validation rules

In [0]:
%python
print("Orders:", orders_df.count())
print("Customers:", customers_df.count())
print("Order Items:", order_items_df.count())
print("Payments:", payments_df.count())
print("Products:", products_df.count())
print("Sellers:", sellers_df.count())
print("Reviews:", reviews_df.count())

## Silver Layer Architecture
### Customer Dimension Table (SCD Type-2)

### Creating Customer Dimension Table (SCD Type-2)

In the Silver layer, we create curated and structured datasets derived from the Bronze layer.  
The `silver_customers_dim` table is designed as a **Slowly Changing Dimension (SCD Type-2)** table to track historical changes in customer attributes.

SCD Type-2 allows us to maintain a full history of changes by storing multiple versions of a record with effective start and end timestamps.

Key fields used for version tracking:

- **start_date** → when the record version becomes active  
- **end_date** → when the record version expires  
- **is_current** → indicates the active record version

In [0]:
%python
spark.sql("""
CREATE TABLE IF NOT EXISTS silver_customers_dim (
customer_id STRING,
customer_unique_id STRING,
customer_city STRING,
customer_state STRING,
start_date TIMESTAMP,
end_date TIMESTAMP,
is_current INT
)
USING DELTA
""")

## Initial Full Load

## Initial Full Load – Customer Dimension (SCD Type 2 Initialization)

The first step in the Silver layer implementation is to initialize the **customer dimension table** using data from the Bronze layer.

This step performs a **full load of the customer dataset** and prepares the table structure required for implementing **Slowly Changing Dimension (SCD) Type-2 logic**.

### Purpose

The objective of this step is to create the first version of the customer dimension table while introducing tracking columns required for historical data management.

### Key Columns Introduced

• **start_date** – Represents the timestamp when the record becomes active.  
• **end_date** – Represents the timestamp when the record becomes inactive. Initially set to `NULL` for active records.  
• **is_current** – Indicates whether the record is the latest version (`1`) or a historical version (`0`).

### Processing Logic

1. Extract customer attributes from the Bronze customer dataset.
2. Add SCD tracking columns (`start_date`, `end_date`, `is_current`).
3. Initialize all records as active (`is_current = 1`).
4. Store the resulting dataset in the Silver layer table **`silver_customers_dim`** using Delta format.

### Outcome

The Silver customer dimension table is created with SCD tracking fields, enabling the pipeline to detect future attribute changes and maintain historical records using SCD Type-2 logic.

In [0]:
%python
initial_customers = customers_df.select(
"customer_id",
"customer_unique_id",
"customer_city",
"customer_state"
).withColumn("start_date", current_timestamp()) \
.withColumn("end_date", lit(None).cast("timestamp")) \
.withColumn("is_current", lit(1))

initial_customers.write.format("delta").mode("overwrite").saveAsTable("silver_customers_dim")

## Detect Customer Attribute Changes

## Detect Customer Attribute Changes

To implement **Slowly Changing Dimension (SCD Type-2)**, the pipeline must identify records where customer attributes have changed.

This step compares incoming Bronze customer records with the **current active records** in the Silver customer dimension table.

### Purpose

The objective is to detect updates in dimension attributes so that historical records can be preserved.

### Processing Logic

1. Retrieve active records from the Silver customer dimension table (`is_current = 1`).
2. Join the Bronze customer dataset with the current Silver records using `customer_id`.
3. Compare relevant attributes such as:

   • `customer_city`  
   • `customer_state`

4. If any attribute difference is detected, the record is flagged as a **changed record**.

### Outcome

The resulting dataset (`changes_df`) contains only the customer records whose attributes have changed.  
These records will be used in the next steps to:

• expire the existing record in the dimension table  
• insert a new version of the record with updated attribute values  

This process ensures that historical customer changes are preserved following **SCD Type-2 principles**.

In [0]:
%python
silver_current = spark.table("silver_customers_dim").filter("is_current = 1")

changes_df = customers_df.alias("src").join(
silver_current.alias("tgt"),
"customer_id"
).where(
(col("src.customer_city") != col("tgt.customer_city")) |
(col("src.customer_state") != col("tgt.customer_state"))
)

## Expire Existing Records


## Expire Existing Records (SCD Type-2 Processing)

After detecting attribute changes in the customer dataset, the existing active record must be expired before inserting the updated version.

This step implements the first stage of **Slowly Changing Dimension Type-2 processing**.

### Purpose

The objective is to preserve historical records by marking the current version of a customer record as inactive when attribute changes are detected.

### Processing Logic

1. Load the customer dimension table stored in the Silver layer.
2. Identify records that have changed using the `changes_df` dataset.
3. Perform a **Delta Lake MERGE operation** to match the changed records with active records in the dimension table.
4. When a match is found, update the existing record by:
   - setting `is_current = 0`
   - updating the `end_date` with the current timestamp.

### Outcome

The previous version of the customer record becomes a **historical record**, enabling the insertion of a new updated version in the next step.

This process ensures that all historical attribute changes are preserved according to **SCD Type-2 principles**.

In [0]:
%python

delta_table = DeltaTable.forName(spark, "silver_customers_dim")
delta_table.alias("tgt").merge(
changes_df.alias("src"),
"tgt.customer_id = src.customer_id AND tgt.is_current = 1"
).whenMatchedUpdate(set={
"is_current": "0",
"end_date": "current_timestamp()"
}).execute()

## Insert New Customers

## Insert New Customers

After processing attribute updates, the pipeline must also detect and insert **new customers** that do not already exist in the customer dimension table.

### Purpose

The objective of this step is to identify new dimension members from the Bronze dataset and insert them into the Silver dimension table.

### Processing Logic

1. Compare the Bronze customer dataset with the active records in the Silver dimension table.
2. Use a **left_anti join** to identify records that exist in the source dataset but not in the target dimension table.
3. Prepare the new records by adding SCD tracking columns:
   - `start_date` – timestamp when the record becomes active
   - `end_date` – initialized as `NULL`
   - `is_current` – set to `1` to indicate the active version
4. Insert the new records into the **silver_customers_dim** table.

### Outcome

New customers are inserted into the dimension table with the appropriate SCD tracking attributes, ensuring that all dimension members are properly captured in the Silver layer.

In [0]:
%python

new_customers = customers_df.alias("src").join(
silver_current.alias("tgt"),
"customer_id",
"left_anti"
)

new_customers_insert = new_customers.select(
"customer_id",
"customer_unique_id",
"customer_city",
"customer_state"
).withColumn("start_date", current_timestamp()) \
.withColumn("end_date", lit(None).cast("timestamp")) \
.withColumn("is_current", lit(1))

new_customers_insert.write.mode("append").saveAsTable("silver_customers_dim")

## Order Business Transformation

## Order Business Transformation

In the Bronze layer, transactional data is stored across multiple operational tables such as orders, order items, and payments.

For analytical workloads, these datasets must be transformed into a structured dataset that combines key transactional attributes.

### Purpose

The objective of this step is to create a unified **order fact dataset** by joining multiple transactional tables.

### Processing Logic

1. Join the **orders dataset** with the **order_items dataset** using `order_id`.
2. Join the resulting dataset with the **payments dataset** to incorporate payment information.
3. Select key business attributes including:

   • `order_id`  
   • `customer_id`  
   • `price`  
   • `freight_value`  
   • `payment_value`  
   • `order_status`  
   • `order_purchase_timestamp`

### Outcome

The resulting dataset (`silver_orders`) represents a structured transactional dataset that can be used for business analytics such as revenue analysis, customer purchase behavior, and operational reporting.

In [0]:
%python

silver_orders = orders_df.join(
order_items_df,
"order_id"
).join(
payments_df,
"order_id"
).select(
"order_id",
"customer_id",
"price",
"freight_value",
"payment_value",
"order_status",
"order_purchase_timestamp"
)

In [0]:
%python
silver_orders.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("silver_orders")

In [0]:
%sql
SELECT *
FROM silver_orders
LIMIT 10

## Business Analytics Example – Revenue per Customer

To demonstrate the analytical capability of the Silver transformation layer, a query is executed on the `silver_orders` dataset to identify the highest revenue-generating customers.

### Query Objective
The goal of this analysis is to calculate the total revenue generated by each customer based on the aggregated payment value of their orders.

### Processing Logic
1. Retrieve transactional records from the `silver_orders` dataset.
2. Group the records by `customer_id`.
3. Calculate the total revenue per customer using `SUM(payment_value)`.
4. Sort the results in descending order to identify the highest revenue contributors.
5. Limit the output to the top 10 customers.

### Business Insight
This analysis helps businesses identify **high-value customers**, enabling targeted marketing strategies, loyalty programs, and customer segmentation initiatives.

In [0]:
SELECT customer_id,
       ROUND(SUM(payment_value),2) AS total_revenue
FROM silver_orders
GROUP BY customer_id
ORDER BY total_revenue DESC
LIMIT 10

## Revenue Aggregation

## Order Revenue Aggregation

To support revenue analytics, the pipeline calculates the total value of each order based on item price and freight charges.

### Purpose

The objective of this step is to compute the total order value by aggregating item-level transactional data.

### Processing Logic

1. Group order item records by `order_id`.
2. Calculate the total value of each order using the formula:

   order_value = price + freight_value

3. Aggregate the results using `SUM()` to compute the total order value per order.

### Outcome

The resulting dataset (`silver_order_revenue`) contains the total value for each order and can be used for revenue analysis, average order value calculations, and financial reporting.

In [0]:
%python

order_revenue = order_items_df.groupBy("order_id").agg(
sum(col("price") + col("freight_value")).alias("order_value")
)

order_revenue.write.format("delta").mode("overwrite").saveAsTable("silver_order_revenue")

## Revenue Validation Query

To validate the order revenue aggregation, a query is executed on the `silver_order_revenue` dataset to calculate the total revenue generated from all orders.

This confirms that the aggregation logic correctly computes the total order value from item-level transactional data.

In [0]:
%sql
SELECT ROUND(SUM(order_value),2) AS total_revenue
FROM silver_order_revenue

## Data Validation Framework

## Data Quality Check – Null `order_id`

This validation ensures that the `order_id` field in the `silver_orders` dataset does not contain null values.
Since `order_id` acts as the primary identifier for order transactions, maintaining its completeness is essential for reliable joins and downstream analytics.

The check confirms that **no records contain null `order_id` values**, ensuring data integrity within the Silver layer.


In [0]:
%python
null_orders = silver_orders.filter(col("order_id").isNull()).count()

In [0]:
%python
print("Null order_id count:", null_orders)

### Duplicate Order ID Check

To maintain data integrity, we validate that each `order_id` in the Silver Orders dataset is unique.  
Duplicate records can lead to incorrect aggregations, revenue miscalculations, and inaccurate analytics.

This check groups the dataset by `order_id` and counts occurrences.  
If the count is greater than 1, it indicates duplicate records that require investigation.

In [0]:
%python
duplicate_orders = silver_orders.groupBy("order_id").count().filter("count > 1")
duplicate_orders.show()

### Payment Value Validation

To ensure business logic integrity, we validate that the `payment_value` field contains only positive values.

In real-world e-commerce systems, payment amounts cannot be negative.  
Negative values may indicate data corruption, incorrect ingestion, or transformation errors.

This check identifies any records where the payment value is less than zero.

In [0]:
%python
invalid_payments = silver_orders.filter(col("payment_value") < 0).count()

In [0]:
%python
print(f"Invalid payment records: {invalid_payments}")

### Order Lifecycle Validation

To maintain logical consistency in the order lifecycle, we verify that the delivery date occurs after the purchase timestamp.

In an e-commerce system, an order cannot be delivered before it is purchased.  
This validation identifies any records where the delivery date is earlier than the purchase time, which would indicate incorrect or corrupted data.

In [0]:
%python
lifecycle_issues = orders_df.filter(
col("order_delivered_customer_date") < col("order_purchase_timestamp")
).count()

In [0]:
%python
print(f"Invalid lifecycle records: {lifecycle_issues}")

## Governance Monitoring

## Validation Audit Logging

After executing validation checks in the Silver layer, the results are recorded in the **validation_audit** table.

### Purpose

The audit table provides governance monitoring by tracking the outcome of data validation rules during pipeline execution.

### Processing Logic

1. Calculate the total number of detected issues from validation checks.
2. Insert a record into the audit table containing:
   - validation rule name
   - number of issues detected
   - execution timestamp

### Outcome

The audit table maintains a historical log of validation results, enabling monitoring, troubleshooting, and governance tracking for data quality processes.

In [0]:
%python
spark.sql(f"""
INSERT INTO validation_audit
VALUES (
'Silver_Data_Validation',
{invalid_payments + lifecycle_issues},
current_timestamp()
)
""")